In [2]:
from apiclient.discovery import build

In [5]:
def video_search(youtube, q='自動化', max_results=50):

    responce = youtube.search().list(
        q = q,
        part = "id,snippet",
        order = "viewCount",
        type = "video",
        maxResults = max_results
    ).execute()

    items_id =[]
    items = responce['items']
    for item in items:
        item_id ={}
        item_id['video_id'] = item['id']['videoId']
        item_id['channel_id'] = item['snippet']['channelId']
        items_id.append(item_id)
    items_id
    df_video = pd.DataFrame(items_id)

    return df_video

In [ ]:
DEVELOPER_KEY = "YOUR KEY"
YOUTUBE_API_SERVICE_NAME = "youtube"
YOUTUBE_API_VERSION = "v3"

youtube = build(YOUTUBE_API_SERVICE_NAME, YOUTUBE_API_VERSION, developerKey=DEVELOPER_KEY)

In [33]:
import pandas as pd

In [36]:
def get_results(df_video, threshold=5000):
    channel_ids = df_video['channel_id'].unique().tolist()

    subscriber_list = youtube.channels().list(
        id = ','.join(channel_ids),
        part = "statistics",
        fields = 'items(id, statistics(subscriberCount))'
    ).execute()

    subscribers = []
    for item in subscriber_list['items']:
        subscriber = {}
        if len(item['statistics']) > 0:    
            subscriber['channel_id'] = item['id']
            subscriber['subscriber_count'] = int(item['statistics']['subscriberCount'])
        else:
            subscriber['channel_id'] = item['id']
        subscribers.append(subscriber)

    df_subscribers = pd.DataFrame(subscribers)

    df = pd.merge(left=df_video, right=df_subscribers, on='channel_id')
    df_extracted = df[df['subscriber_count'] < threshold]
    video_ids = df_extracted['video_id'].tolist()
    videos_list = youtube.videos().list(
        id = ','.join(video_ids),
        part = 'snippet,statistics',
        fields = 'items(id, snippet(title), statistics(viewCount))'
    ).execute()

    videos_info = []

    items = videos_list['items']
    for item in items:
        video_info = {}
        video_info['video_id'] = item['id']
        video_info['title'] = item['snippet']['title']
        video_info['view_count'] = item['statistics']['viewCount']
        videos_info.append(video_info)
    df_videos_info = pd.DataFrame(videos_info)
    results = pd.merge(left=df_extracted, right=df_videos_info, on='video_id')
    results = results.loc[:,['video_id', 'title', 'view_count', 'subscriber_count', 'channel_id']]
    return results

In [39]:
df_video = video_search(youtube, q='Python 自動化', max_results=50)
results = get_results(df_video, threshold=1000)

In [40]:
results[:3]

,video_id,title,view_count,subscriber_count,channel_id
0,bU7-a7IcJsY,【ひろゆき】経理の業務、PythonでExcel効率化したい #ひろゆき #ひろゆき切り抜き,199816,290,UCZXRMLFAzFeQf-Qm5QnLWyA
